In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
texts = [
    "i love this movie",
    "this film was terrible",
    "amazing acting and story",
    "worst experience ever"
]

labels = torch.tensor([1, 0, 1, 0])  # binary classification

In [5]:
from collections import Counter

def build_vocab(texts):
    words = []
    for text in texts:
        words.extend(text.split())

    vocab = {word: i+1 for i,word in enumerate(set(words))}
    return vocab

vocab = build_vocab(texts)

In [7]:
def encode(text, vocab, max_len):
    tokens = [vocab[word] for word in text.split()]
    tokens = tokens[:max_len]
    tokens += [0] * (max_len - len(tokens))  # padding
    return tokens

max_len = 10
X = torch.tensor([encode(text, vocab, max_len) for text in texts])

In [9]:
X

tensor([[ 9,  5,  7,  8,  0,  0,  0,  0,  0,  0],
        [ 7,  2,  1,  4,  0,  0,  0,  0,  0,  0],
        [12,  3,  6, 14,  0,  0,  0,  0,  0,  0],
        [10, 11, 13,  0,  0,  0,  0,  0,  0,  0]])

In [8]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2) * (-torch.log(torch.tensor(10000.0)) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [9]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim, num_layers, max_len):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoding = PositionalEncoding(embed_dim, max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.fc = nn.Linear(embed_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)                  # (batch, seq, embed)
        x = self.pos_encoding(x)

        x = self.transformer(x)               # self-attention happens here

        x = x.mean(dim=1)                     # pooling
        x = self.fc(x)
        return self.sigmoid(x)

In [10]:
vocab_size = len(vocab) + 1
model = TransformerClassifier(
    vocab_size=vocab_size,
    embed_dim=32,
    num_heads=2,
    ff_dim=64,
    num_layers=2,
    max_len=max_len
)

In [11]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [12]:
epochs = 10

for epoch in range(epochs):
    model.train()

    outputs = model(X).squeeze()
    loss = criterion(outputs, labels.float())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.7239
Epoch 2, Loss: 0.6920
Epoch 3, Loss: 0.6893
Epoch 4, Loss: 0.6726
Epoch 5, Loss: 0.6679
Epoch 6, Loss: 0.6475
Epoch 7, Loss: 0.6170
Epoch 8, Loss: 0.6063
Epoch 9, Loss: 0.5739
Epoch 10, Loss: 0.5658


In [13]:
model.eval()
with torch.no_grad():
    preds = model(X).squeeze()
    predictions = (preds > 0.5).int()

print("Predictions:", predictions)

Predictions: tensor([1, 0, 1, 0], dtype=torch.int32)
